In [3]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [4]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по лошадям v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Лошади
145,АКТЮБИНСКАЯ ОБЛАСТЬ,2016-07-01,613.56
2002,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2023-12-01,1520.56
1489,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2022-06-01,617.37
441,АТЫРАУСКАЯ ОБЛАСТЬ,2020-01-01,566.20
31,АКМОЛИНСКАЯ ОБЛАСТЬ,2017-08-01,869.58
720,ГАСТАНА,2018-06-01,3.13
636,ГАЛМАТЫ,2015-03-01,0.80
686,ГАСТАНА,2015-08-01,2.95
1431,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2017-08-01,585.81
1898,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2015-04-01,279.55


In [5]:
regions = df['Регион'].unique()
target   = "Лошади"
horizon  = 3
epsilon = 1e-6

In [6]:
first_test = pd.to_datetime("2024-08-01")
last_possible = df["Период"].max() - pd.DateOffset(months=horizon-1)
test_starts = pd.date_range(first_test, last_possible, freq="MS")

## Holt-Winter's (log)

In [7]:
results_hw = []
for region in regions:
    ts = (df[df["Регион"] == region]
          .set_index("Период")[target]
          .dropna()
          .sort_index())
    if len(ts) < 24:
        print(f"{region}: всего {len(ts)} мес. — сезонный Holt-Winter's невозможен.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        # формируем train / test
        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]

        # пропускаем, если недостаточно данных или неполный test
        if len(train) < 24 or len(test) < horizon:
            continue

        # обучаем модель
        train_log = np.log1p(train)

        hw_log = ExponentialSmoothing(
            train_log,
            seasonal="add",
            seasonal_periods=12
        ).fit(optimized=True)

        # прогноз и метрики
        fc_log = hw_log.forecast(horizon)
        fc = np.expm1(fc_log) 
        # fc   = model.forecast(horizon)
        rmse = np.sqrt(mean_squared_error(test, fc))
        mae  = mean_absolute_error(test, fc)
        mape = (np.abs((test - fc) / test).mean()) * 100

        results_hw.append({
            "Регион":      region,
            "Test start":  test_start.strftime("%Y-%m"),
            "Test end":    test_end.strftime("%Y-%m"),
            "Forecast":    [x.round(2) for x in list(fc.values)],
            "Actual":      [y.round(2) for y in list(test.values)],
            "RMSE":        rmse,
            "MAE":         mae,
            "MAPE_%":      mape
        })

# 4) Усреднение по всем скользящим окнам для каждого региона
res_hw = pd.DataFrame(results_hw)
res_hw.to_excel("results/Лошади - Результаты прогнозов ХВ v2.xlsx", index=False)
print("Результаты прогнозов HW на 3 месяца:")

display(res_hw)

final_hw = (
    res_hw
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_hw.to_excel("results/Лошади - Результаты прогнозов ХВ средние v2.xlsx", index=False)
print("Средние метрики Holt–Winter's по регионам (rolling-3):")
display(final_hw)

Результаты прогнозов HW на 3 месяца:


,Регион,Test start,Test end,Forecast,Actual,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"[931.67, 1304.68, 981.26]","[995.56, 1304.96, 1013.53]",41.33,32.15,3.21
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"[1311.76, 986.56, 2054.63]","[1304.96, 1013.53, 2212.19]",92.38,63.78,3.44
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"[986.05, 2053.6, 1821.92]","[1013.53, 2212.19, 1876.09]",98.05,80.08,4.26
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"[2058.87, 1826.05, 1645.96]","[2212.19, 1876.09, 1610.32]",95.36,79.67,3.94
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"[1837.93, 1657.53, 992.25]","[1876.09, 1610.32, 1067.0]",55.60,53.37,3.99
...,...,...,...,...,...,...,...,...
185,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"[3585.02, 3970.47, 4253.01]","[2598.76, 2732.23, 2726.66]",1269.61,1250.29,46.42
186,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"[3227.68, 3465.65, 2983.61]","[2732.23, 2726.66, 2579.69]",564.13,546.12,20.30
187,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"[3107.2, 2679.17, 1974.26]","[2726.66, 2579.69, 1902.11]",230.88,184.06,7.20
188,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"[2449.93, 1807.7, 2641.67]","[2579.69, 1902.11, 2208.35]",266.78,219.16,9.87


Средние метрики Holt–Winter's по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,86.94,76.92,5.75
1,АКТЮБИНСКАЯ ОБЛАСТЬ,279.50,225.85,9.32
2,АЛМАТИНСКАЯ ОБЛАСТЬ,209.25,152.38,8.89
3,АТЫРАУСКАЯ ОБЛАСТЬ,110.69,92.39,8.89
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,349.13,293.32,18.27
5,ГАСТАНА,0.96,0.81,25.41
6,ГШЫМКЕНТ,62.78,44.93,49.18
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,50.52,42.54,3.12
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,111.04,83.11,5.29
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,210.08,185.21,10.47


## SARIMA

In [8]:
results_sarima = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    ts = ts + epsilon
    ts_log = np.log(ts)

    if len(ts_log) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. для авто-ARIMA, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train_log = ts_log[ts_log.index < test_start]
        test_log  = ts_log[(ts_log.index >= test_start) & (ts_log.index <= test_end)]
        if len(train_log) < 12 + horizon or len(test_log) < horizon:
            continue

        # автоподбор на лог-данных
        use_seasonal = len(train_log) >= 2 * 12

        sarima_log = auto_arima(
            train_log,
            seasonal=use_seasonal,
            m=12 if use_seasonal else 1,
            D=1 if use_seasonal else 0,      # фиксируем порядок сезонной разности
            seasonal_test=None,               # пропустить nsdiffs
            boxcox=True,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )
      
        # прогноз в лог-шкале
        fc_log = sarima_log.predict(n_periods=horizon, return_conf_int=False)

        # возвращаем прогноз в исходные единицы
        fc = np.exp(fc_log) - epsilon
        actual = np.exp(test_log.values) - epsilon  # но exp(log(x)) == x

        # метрики на исходном уровне
        rmse = np.sqrt(mean_squared_error(actual, fc))
        mae  = mean_absolute_error(actual, fc)
        mape = (np.abs((actual - fc) / actual).mean()) * 100

        results_sarima.append({
            "Регион":         region,
            "Test start":     test_start.strftime("%Y-%m"),
            "Test end":       test_end.strftime("%Y-%m"),
            "order":          sarima_log.order,
            "seasonal_order": sarima_log.seasonal_order,
            "RMSE":           round(rmse,2),
            "MAE":            round(mae,2),
            "MAPE_%":         round(mape,2),
            "Forecast":       [round(x,2) for x in fc],
            "Actual":         [round(y,2) for y in actual]
        })
# формируем DataFrame с результатами
res_sarima = pd.DataFrame(results_sarima)
res_sarima.to_excel("results/Лошади - Результаты прогнозов SARIMA v2.xlsx", index=False)
print("Результаты прогнозов SARIMA на 3 месяца:")

display(res_sarima)

final_sarima = (
    res_sarima
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_sarima.to_excel("results/Лошади - Результаты прогнозов SARIMA средние v2.xlsx", index=False)
print("Средние метрики SARIMA по регионам (rolling-3):")
display(final_sarima)

Результаты прогнозов SARIMA на 3 месяца:


,Регион,Test start,Test end,order,seasonal_order,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"(0, 0, 0)","(1, 1, 0, 12)",35.72,29.12,2.77,"[937.45, 1323.07, 1002.4]","[995.56, 1304.96, 1013.53]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"(0, 0, 0)","(1, 1, 0, 12)",82.62,56.93,2.96,"[1323.73, 1002.98, 2070.72]","[1304.96, 1013.53, 2212.19]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"(0, 0, 0)","(1, 1, 0, 12)",89.06,70.70,3.55,"[1002.82, 2070.27, 1816.62]","[1013.53, 2212.19, 1876.09]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"(0, 0, 0)","(1, 1, 0, 12)",93.13,83.37,4.21,"[2070.41, 1816.82, 1659.39]","[2212.19, 1876.09, 1610.32]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"(0, 0, 0)","(1, 1, 0, 12)",57.17,56.99,4.02,"[1818.26, 1661.44, 1004.98]","[1876.09, 1610.32, 1067.0]"
...,...,...,...,...,...,...,...,...,...,...
185,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"(1, 0, 0)","(0, 1, 0, 12)",162.54,128.30,4.86,"[2856.93, 2621.06, 2711.09]","[2598.76, 2732.23, 2726.66]"
186,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"(1, 0, 0)","(0, 1, 0, 12)",252.24,225.80,8.53,"[2496.98, 2643.03, 2221.17]","[2732.23, 2726.66, 2579.69]"
187,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"(1, 0, 0)","(0, 1, 0, 12)",185.02,140.47,5.78,"[2761.67, 2270.7, 1979.53]","[2726.66, 2579.69, 1902.11]"
188,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"(1, 0, 0)","(0, 1, 0, 12)",221.30,195.95,8.34,"[2256.58, 1973.32, 2401.87]","[2579.69, 1902.11, 2208.35]"


Средние метрики SARIMA по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,78.12,68.16,5.08
1,АКТЮБИНСКАЯ ОБЛАСТЬ,211.18,169.39,7.20
2,АЛМАТИНСКАЯ ОБЛАСТЬ,213.67,163.33,10.08
3,АТЫРАУСКАЯ ОБЛАСТЬ,76.78,67.14,7.69
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,275.19,218.21,12.36
5,ГАСТАНА,0.85,0.70,22.23
6,ГШЫМКЕНТ,78.56,64.55,91.95
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,58.00,50.01,3.74
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,95.68,69.10,4.89
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,137.70,104.79,5.95


## Facebook Prophet

In [9]:
results_prophet = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    if len(ts) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. данных, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]
        if len(train) < 12 + horizon or len(test) < horizon:
            continue

        # Подготовка данных для Prophet
#         df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
# # подготовка для одного региона
        df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
        df_prophet["y"] = np.log(df_prophet["y"] + epsilon)

        m = Prophet()
        m.fit(df_prophet)

        future = m.make_future_dataframe(periods=horizon, freq="MS")
        forecast = m.predict(future)

        # берем только прогнозные точки
        yhat_log = forecast["yhat"].values[-horizon:]
        fc = np.exp(yhat_log) - epsilon

        # m = Prophet()
        # m.fit(df_prophet)

        # # Создаем DataFrame будущих дат и делаем прогноз
        # # future = m.make_future_dataframe(periods=horizon, freq="MS")
        # # forecast = m.predict(future)

        # # Отбираем только наши горизонты
        # fc = forecast.set_index("ds")["yhat"].loc[test.index].values
        actual = test.values

        # Расчет метрик
        rmse  = np.sqrt(mean_squared_error(actual, fc))
        mae   = mean_absolute_error(actual, fc)
        mape  = (np.abs((actual - fc) / actual).mean()) * 100

        results_prophet.append({
            "Регион":     region,
            "Test start": test_start.strftime("%Y-%m"),
            "Test end":   test_end.strftime("%Y-%m"),
            "RMSE":       round(rmse, 2),
            "MAE":        round(mae, 2),
            "MAPE_%":     round(mape, 2),
            "Forecast":   [round(x, 2) for x in fc],
            "Actual":     [round(x, 2) for x in actual]
        })

# Собираем результаты в DataFrame
res_prophet = pd.DataFrame(results_prophet)
res_prophet.to_excel("results/Лошади - Результаты прогнозов Prophet v2.xlsx", index=False)
print("Результаты прогнозов Prophet на 3 месяца:")
display(res_prophet)

final_prophet = (
    res_prophet
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_prophet.to_excel("results/Лошади - Результаты прогнозов Prophet средние v2.xlsx", index=False)
print("Средние метрики Prophet по регионам (rolling-3):")
display(final_prophet)


16:23:31 - cmdstanpy - INFO - Chain [1] start processing
16:23:32 - cmdstanpy - INFO - Chain [1] done processing
16:23:32 - cmdstanpy - INFO - Chain [1] start processing
16:23:32 - cmdstanpy - INFO - Chain [1] done processing
16:23:33 - cmdstanpy - INFO - Chain [1] start processing
16:23:33 - cmdstanpy - INFO - Chain [1] done processing
16:23:33 - cmdstanpy - INFO - Chain [1] start processing
16:23:33 - cmdstanpy - INFO - Chain [1] done processing
16:23:33 - cmdstanpy - INFO - Chain [1] start processing
16:23:33 - cmdstanpy - INFO - Chain [1] done processing
16:23:33 - cmdstanpy - INFO - Chain [1] start processing
16:23:34 - cmdstanpy - INFO - Chain [1] done processing
16:23:34 - cmdstanpy - INFO - Chain [1] start processing
16:23:34 - cmdstanpy - INFO - Chain [1] done processing
16:23:34 - cmdstanpy - INFO - Chain [1] start processing
16:23:34 - cmdstanpy - INFO - Chain [1] done processing
16:23:34 - cmdstanpy - INFO - Chain [1] start processing
16:23:35 - cmdstanpy - INFO - Chain [1]

Результаты прогнозов Prophet на 3 месяца:


,Регион,Test start,Test end,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,13.37,10.78,0.99,"[997.15, 1295.1, 992.64]","[995.56, 1304.96, 1013.53]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,78.91,54.43,2.90,"[1296.46, 993.7, 2077.23]","[1304.96, 1013.53, 2212.19]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,83.87,66.51,3.49,"[992.38, 2074.28, 1916.57]","[1013.53, 2212.19, 1876.09]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,82.24,60.64,2.87,"[2077.22, 1921.58, 1608.86]","[2212.19, 1876.09, 1610.32]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,30.34,23.31,1.46,"[1926.3, 1615.32, 1052.29]","[1876.09, 1610.32, 1067.0]"
...,...,...,...,...,...,...,...,...
185,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,255.18,248.19,9.25,"[2342.54, 2904.11, 3043.13]","[2598.76, 2732.23, 2726.66]"
186,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,236.78,213.49,7.89,"[2925.23, 3074.55, 2679.28]","[2732.23, 2726.66, 2579.69]"
187,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,206.62,173.62,6.98,"[3058.23, 2663.74, 2007.34]","[2726.66, 2579.69, 1902.11]"
188,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,408.88,289.07,13.16,"[2653.02, 1998.18, 2906.17]","[2579.69, 1902.11, 2208.35]"


Средние метрики Prophet по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,99.51,79.54,6.00
1,АКТЮБИНСКАЯ ОБЛАСТЬ,269.24,222.54,9.90
2,АЛМАТИНСКАЯ ОБЛАСТЬ,727.29,594.89,31.63
3,АТЫРАУСКАЯ ОБЛАСТЬ,164.37,144.20,15.10
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,506.75,456.56,25.99
5,ГАСТАНА,1.29,1.04,29.14
6,ГШЫМКЕНТ,81.15,60.99,75.19
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,115.45,96.40,6.41
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,178.55,137.43,10.76
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,569.38,473.76,23.86


In [10]:
# Переименуем колонки с MAPE, чтобы было понятно, к какому методу относятся
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW"})
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA"})
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet"})

# Мёрджим по региону
summary = (
    hw[["Регион", "MAPE_HW"]]
    .merge(sar[["Регион", "MAPE_SARIMA"]], on="Регион")
    .merge(pr[["Регион", "MAPE_Prophet"]], on="Регион")
)

# Определяем для каждой строки, какой столбец MAPE минимален
# idxmin вернёт название столбца с минимальным значением
summary["Best_method"] = summary[["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]] \
                           .idxmin(axis=1) \
                           .str.replace("MAPE_","")  # убираем префикс для красоты

# Если нужно, можно сразу отсортировать
# summary = summary.sort_values("Best_method")

# допустим, у вас уже есть summary
summary = summary.round({
    "MAPE_HW": 2,
    "MAPE_SARIMA": 2,
    "MAPE_Prophet": 2
})

# Готово!
print(summary.to_string(index=False))
summary.to_excel("results/Лошади - Лучшие модели v2.xlsx", index=False)


                        Регион  MAPE_HW  MAPE_SARIMA  MAPE_Prophet Best_method
           АКМОЛИНСКАЯ ОБЛАСТЬ     5.75         5.08          6.00      SARIMA
           АКТЮБИНСКАЯ ОБЛАСТЬ     9.32         7.20          9.90      SARIMA
           АЛМАТИНСКАЯ ОБЛАСТЬ     8.89        10.08         31.63          HW
            АТЫРАУСКАЯ ОБЛАСТЬ     8.89         7.69         15.10      SARIMA
ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ    18.27        12.36         25.99      SARIMA
                       ГАСТАНА    25.41        22.23         29.14      SARIMA
                      ГШЫМКЕНТ    49.18        91.95         75.19          HW
            ЖАМБЫЛСКАЯ ОБЛАСТЬ     3.12         3.74          6.41          HW
 ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ     5.29         4.89         10.76      SARIMA
        КАРАГАНДИНСКАЯ ОБЛАСТЬ    10.47         5.95         23.86      SARIMA
          КОСТАНАЙСКАЯ ОБЛАСТЬ     8.34         9.30          8.68          HW
        КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ    12.51        12.07

In [11]:
#FOLDER = Path("results")  # папка, где лежат файлы
FILE_HW      = "results/Лошади - Результаты прогнозов ХВ средние v2.xlsx"
FILE_SARIMA  = "results/Лошади - Результаты прогнозов SARIMA средние v2.xlsx"
FILE_PROPHET = "results/Лошади - Результаты прогнозов Prophet средние v2.xlsx"


# === Загрузка исходных таблиц ===
final_hw      = pd.read_excel(FILE_HW)
final_sarima  = pd.read_excel(FILE_SARIMA)
final_prophet = pd.read_excel(FILE_PROPHET)

# Ожидаемые столбцы: 'Регион', 'MAPE_%', 'MAE' (и/или 'RMSE')
# Переименуем для прозрачности
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW", "MAE": "MAE_HW"})[["Регион","MAPE_HW","MAE_HW"]]
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA", "MAE": "MAE_SARIMA"})[["Регион","MAPE_SARIMA","MAE_SARIMA"]]
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet", "MAE": "MAE_Prophet"})[["Регион","MAPE_Prophet","MAE_Prophet"]]

# === Объединяем по региону ===
summary = (
    hw.merge(sar, on="Регион", how="inner")
      .merge(pr,  on="Регион", how="inner")
)


In [12]:
THRESHOLD_MAPE = 1000.0  # порог

mape_cols = ["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]
mae_cols  = ["MAE_HW","MAE_SARIMA","MAE_Prophet"]

# 1) Приведём метрики к числам (на всякий случай ещё раз)
for c in mape_cols + mae_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

def choose_best_simple(row):
    # Берём числовые серии и подменяем NaN на +inf, чтобы .idxmin() стабильно работал
    mape_s = row[mape_cols].astype(float).fillna(np.inf)
    mae_s  = row[mae_cols].astype(float).fillna(np.inf)

    min_mape = mape_s.min()

    # Если все MAPE были NaN -> min = +inf
    if np.isinf(min_mape):
        criterion = "MAE"
        winner_col = mae_s.idxmin()
    elif min_mape <= THRESHOLD_MAPE:
        criterion = "MAPE"
        winner_col = mape_s.idxmin()
    else:
        criterion = "MAE"
        winner_col = mae_s.idxmin()

    method = winner_col.split("_")[-1]  # HW / SARIMA / Prophet

    return pd.Series({
        "Best_method": method,
        "Best_criterion": criterion,
        "Best_MAPE": float(mape_s.replace(np.inf, np.nan).min()),
        "Best_MAE": float(mae_s.replace(np.inf, np.nan).min())
    })

best = summary.apply(choose_best_simple, axis=1)

result = pd.concat([summary, best], axis=1)

# Округление и сохранение
for c in mape_cols + mae_cols + ["Best_MAPE","Best_MAE"]:
    result[c] = result[c].round(2)

# Если у вас есть переменная OUT_FILE — используйте её. Иначе:
OUT_FILE = "results/Лошади - Лучшие модели (MAPE_then_MAE) v2.xlsx"
result.sort_values(["Best_method","Регион"]).to_excel(OUT_FILE, index=False)

result

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,5.75,76.92,5.08,68.16,6.00,79.54,SARIMA,MAPE,5.08,68.16
1,АКТЮБИНСКАЯ ОБЛАСТЬ,9.32,225.85,7.20,169.39,9.90,222.54,SARIMA,MAPE,7.20,169.39
2,АЛМАТИНСКАЯ ОБЛАСТЬ,8.89,152.38,10.08,163.33,31.63,594.89,HW,MAPE,8.89,152.38
3,АТЫРАУСКАЯ ОБЛАСТЬ,8.89,92.39,7.69,67.14,15.10,144.20,SARIMA,MAPE,7.69,67.14
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,18.27,293.32,12.36,218.21,25.99,456.56,SARIMA,MAPE,12.36,218.21
5,ГАСТАНА,25.41,0.81,22.23,0.70,29.14,1.04,SARIMA,MAPE,22.23,0.70
6,ГШЫМКЕНТ,49.18,44.93,91.95,64.55,75.19,60.99,HW,MAPE,49.18,44.93
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,3.12,42.54,3.74,50.01,6.41,96.40,HW,MAPE,3.12,42.54
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,5.29,83.11,4.89,69.10,10.76,137.43,SARIMA,MAPE,4.89,69.10
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,10.47,185.21,5.95,104.79,23.86,473.76,SARIMA,MAPE,5.95,104.79
